# 04 — DQ Metrics Over Time

Turn the quarantine table into a longitudinal **`dq_results`** Delta table that
records, for each run:

- `run_id`, `run_time`, `table_name`
- `rule_name`, `severity`, `failing_rows`, `total_rows`, `pass_rate`

This is the shape Lakeview dashboards / Lakeflow alerts expect — one row per
(run, rule) so you can chart pass-rate-by-rule over time and SLA against it.

In [1]:
from databricks.connect import DatabricksSession
from dotenv import load_dotenv
from datetime import datetime, timezone
import os, uuid

load_dotenv()
spark = DatabricksSession.builder.serverless().getOrCreate()

UC_CATALOG = os.getenv("UC_CATALOG", "alexander_booth")
UC_SCHEMA  = os.getenv("UC_SCHEMA",  "dqx_baseball")

BRONZE     = f"{UC_CATALOG}.{UC_SCHEMA}_bronze.pitches_raw"
QUARANTINE = f"{UC_CATALOG}.{UC_SCHEMA}_quarantine.pitches_quarantine"
DQ_RESULTS = f"{UC_CATALOG}.{UC_SCHEMA}_silver.dq_results"

RUN_ID   = str(uuid.uuid4())
RUN_TIME = datetime.now(timezone.utc).isoformat()
print(f"run_id={RUN_ID}  run_time={RUN_TIME}")

run_id=e710dbde-b3f9-4023-bbfd-0524acfa9261  run_time=2026-05-08T14:49:00.803941+00:00


In [2]:
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {DQ_RESULTS} (
        run_id        STRING,
        run_time      TIMESTAMP,
        table_name    STRING,
        rule_name     STRING,
        severity      STRING,
        failing_rows  BIGINT,
        total_rows    BIGINT,
        pass_rate     DOUBLE
    )
    USING DELTA
    CLUSTER BY (run_time)
    COMMENT 'Per-run DQX rule outcomes from the dqx-baseball-data-quality demo.'
""")
print(f"Ready: {DQ_RESULTS}")

Ready: alexander_booth.dqx_baseball_silver.dq_results


In [3]:
# Roll up failures from quarantine: explode _errors and _warnings, count by rule.
spark.sql(f"""
    INSERT INTO {DQ_RESULTS}
    WITH bronze_total AS (
        SELECT COUNT(*) AS total FROM {BRONZE}
    ),
    failures AS (
        SELECT e.name AS rule_name, 'error' AS severity
        FROM {QUARANTINE} LATERAL VIEW explode(_errors) AS e
        UNION ALL
        SELECT w.name, 'warn'
        FROM {QUARANTINE} LATERAL VIEW explode(_warnings) AS w
    ),
    rolled AS (
        SELECT rule_name, severity, COUNT(*) AS failing_rows
        FROM failures
        GROUP BY rule_name, severity
    )
    SELECT
        '{RUN_ID}'                                AS run_id,
        TIMESTAMP'{RUN_TIME}'                     AS run_time,
        '{BRONZE}'                                AS table_name,
        r.rule_name,
        r.severity,
        r.failing_rows,
        bt.total                                  AS total_rows,
        1.0 - (r.failing_rows / bt.total)         AS pass_rate
    FROM rolled r CROSS JOIN bronze_total bt
""")
print("Inserted run results.")

Inserted run results.


In [4]:
# What this run looked like, sorted worst-first
spark.sql(f"""
    SELECT rule_name, severity, failing_rows, total_rows,
           ROUND(pass_rate * 100, 2) AS pass_pct
    FROM {DQ_RESULTS}
    WHERE run_id = '{RUN_ID}'
    ORDER BY pass_rate ASC
""").show(truncate=False)

+---------------------------+--------+------------+----------+--------+
|rule_name                  |severity|failing_rows|total_rows|pass_pct|
+---------------------------+--------+------------+----------+--------+
|batting_avg_not_in_range   |error   |1342        |50000     |97.32   |
|pitch_speed_in_range       |error   |1295        |50000     |97.41   |
|inning_not_in_range        |warn    |693         |50000     |98.61   |
|batter_id_is_null_or_empty |error   |677         |50000     |98.65   |
|pitch_type_in_enum         |warn    |671         |50000     |98.66   |
|not_game_date_current_date |error   |663         |50000     |98.67   |
|pitcher_id_is_null_or_empty|error   |641         |50000     |98.72   |
|at_bat_id_is_not_unique    |error   |100         |50000     |99.8    |
+---------------------------+--------+------------+----------+--------+



In [5]:
# Cumulative view across all runs in the table — drop into a Lakeview line chart
# (one line per rule_name, x = run_time, y = pass_pct).
spark.sql(f"""
    SELECT run_time, rule_name, severity,
           ROUND(pass_rate * 100, 2) AS pass_pct
    FROM {DQ_RESULTS}
    ORDER BY run_time DESC, pass_rate ASC
""").show(20, truncate=False)

+--------------------------+---------------------------+--------+--------+
|run_time                  |rule_name                  |severity|pass_pct|
+--------------------------+---------------------------+--------+--------+
|2026-05-08 14:49:00.803941|batting_avg_not_in_range   |error   |97.32   |
|2026-05-08 14:49:00.803941|pitch_speed_in_range       |error   |97.41   |
|2026-05-08 14:49:00.803941|inning_not_in_range        |warn    |98.61   |
|2026-05-08 14:49:00.803941|batter_id_is_null_or_empty |error   |98.65   |
|2026-05-08 14:49:00.803941|pitch_type_in_enum         |warn    |98.66   |
|2026-05-08 14:49:00.803941|not_game_date_current_date |error   |98.67   |
|2026-05-08 14:49:00.803941|pitcher_id_is_null_or_empty|error   |98.72   |
|2026-05-08 14:49:00.803941|at_bat_id_is_not_unique    |error   |99.8    |
+--------------------------+---------------------------+--------+--------+



## Where to take this next

- **Schedule** notebooks 01 → 03 → 04 as a Databricks Job and you'll accumulate
  one row per (run, rule) per execution — chart-ready.
- **Alert** with Lakeflow on `pass_rate < 0.95` for any `severity = 'error'` rule.
- **Swap to streaming**: DQX's `apply_checks_and_split` works on Structured Streaming
  DataFrames unchanged — point at a streaming bronze and write quarantine via
  `foreachBatch`.
- **DLT / Lakeflow Pipelines**: `DQDltGenerator.generate_dlt_rules(profiles, language='Python')`
  in notebook 02 emits `@dlt.expect_all` blocks you can paste into a pipeline.